# DataFrame Optimised — Naive Bayes Classifier

**DataFrame implementation with three targeted optimisations over the baseline.**

All logic is identical to `naive_bayes_df_baseline.ipynb` except for
these three changes (each marked with `# OPTIMISATION N`):

1. Replace UDFs with built-in Spark functions (`F.concat`, `F.log`, etc.)
   so Catalyst can inspect and optimise the full query plan.
2. Call `.persist()` on the feature-counts DataFrame after training
   to avoid recomputing the shuffle if referenced again.
3. Call `.repartition(n)` at data load time to tune parallelism for the cluster.

These optimisations are symmetric to the RDD optimisations, making the
RDD-vs-DataFrame comparison apples-to-apples.

In [ ]:
# Cell 1 — Imports and SparkSession setup
import time
import math
import sys
import os

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import StringType

sys.path.insert(0, os.path.abspath('..'))

from core.naive_bayes import compute_log_probs, evaluate
from data.loader import load_car_dataframe, FEATURE_COLS, LABEL_COL

# predict is defined locally so cloudpickle serialises it by value (bytecode).
# Functions imported from external modules are serialised by reference — workers
# would need to import core/, which requires the module to be present on every
# worker node. Defining predict here avoids that dependency entirely.
def predict(log_prob_table, test_point):
    best_class = None
    best_score = float('-inf')
    for class_label in log_prob_table['classes']:
        score = log_prob_table['log_class_probs'][class_label]
        for feat_idx, value in enumerate(test_point):
            key = f'feat_{feat_idx}_{value}_{class_label}'
            if key in log_prob_table['log_feature_probs']:
                score += log_prob_table['log_feature_probs'][key]
            else:
                score += log_prob_table['fallback_log_probs'].get(
                    (feat_idx, class_label), math.log(1e-10)
                )
        if score > best_score:
            best_score = score
            best_class = class_label
    return best_class

# TODO (Dataproc): Remove .master() — the cluster provides its own master URL.
spark = (
    SparkSession.builder
    .master('local[*]')
    .appName('NaiveBayes-DF-Optimised')
    .getOrCreate()
)
sc = spark.sparkContext
sc.setLogLevel('WARN')
print('SparkSession ready.')

In [ ]:
# Cell 2 — Load data
#
# OPTIMISATION 3: repartition at load time
# The number of partitions controls how many parallel tasks Spark can run.
# Too few partitions under-utilises the cluster; too many causes scheduling overhead.
# A common heuristic is 2-4 partitions per CPU core on the cluster.
#
# TODO: Replace None with your dataset path.
#   Local      : '/Users/you/data/car.data'
#   Databricks : 'dbfs:/FileStore/car.data'
# TODO: Set NUM_PARTITIONS based on your Databricks cluster size.
#   Rule of thumb: (number of executor cores) * 2
#   On a single-node local machine, 4 is a reasonable default.
DATA_PATH       = None
NUM_PARTITIONS  = 4  # TODO: tune for your cluster

train_df, test_df = load_car_dataframe(spark, filepath=DATA_PATH)

# Repartition after loading so all downstream operations benefit from
# better parallelism. We do this once here rather than per-transformation.
train_df = train_df.repartition(NUM_PARTITIONS)
test_df  = test_df.repartition(NUM_PARTITIONS)

print(f'Train rows      : {train_df.count()}')
print(f'Test rows       : {test_df.count()}')
print(f'Train partitions: {train_df.rdd.getNumPartitions()}')

print('Class distribution in training data:')
train_df.groupBy(LABEL_COL).count().orderBy('count', ascending=False).show()

In [ ]:
# Cell 3 — Training phase
#
# OPTIMISATION 1: replace Python UDFs with built-in Spark functions
# UDFs are opaque to Catalyst — Spark treats them as black boxes and cannot
# merge them with adjacent operations, push predicates through them, or
# use vectorised (columnar) execution. Built-in functions like F.concat()
# and F.lit() are part of Catalyst's expression tree, so the optimizer
# can combine and reorder them freely. This alone can give a significant speedup.

t_train_start = time.time()

# Count class occurrences using groupBy — same as baseline.
# Schema: [label, class_count]
class_counts_df = train_df.groupBy(LABEL_COL).agg(F.count('*').alias('class_count'))

# Unpivot all feature columns into one tall DataFrame.
# For each feature, select the key as a Catalyst expression using F.concat().
# F.concat() is a built-in function — Catalyst can inspect and optimise it,
# unlike the UDF used in the baseline.
#
# Schema of each key_df: [key]
# Example key value: 'feat_0_low_unacc'
key_dfs = []
for feat_idx, col_name in enumerate(FEATURE_COLS):
    key_df = train_df.select(
        F.concat(
            F.lit(f'feat_{feat_idx}_'),
            F.col(col_name),
            F.lit('_'),
            F.col(LABEL_COL)
        ).alias('key')
    )
    key_dfs.append(key_df)

# Union all feature key DataFrames into one tall DataFrame.
# Schema: [key]
all_keys_df = key_dfs[0]
for df in key_dfs[1:]:
    all_keys_df = all_keys_df.union(df)

# Count occurrences of each key — the REDUCE step.
# Schema: [key, count]
feature_counts_df = all_keys_df.groupBy('key').agg(F.count('*').alias('count'))

# OPTIMISATION 2: persist the feature-counts DataFrame
# DataFrames are also lazily evaluated — every action triggers a full recomputation
# from source. If feature_counts_df is used in more than one action (collect() +
# any subsequent query), persist() materialises it after the first action so
# subsequent uses read from cached memory instead of rerunning the groupBy + union.
feature_counts_df.persist()

train_time = time.time() - t_train_start
print(f'[TIMING] Training: {train_time:.4f}s')
print(f'Unique feature keys: {feature_counts_df.count()}')
feature_counts_df.show(5)

In [ ]:
# Cell 4 — Probability computation
#
# Identical to the baseline — compute_log_probs() is shared pure Python.
# The persist() in Cell 3 means this collect() reads from cache.

class_counts_dict   = {row[LABEL_COL]: row['class_count'] for row in class_counts_df.collect()}
feature_counts_dict = {row['key']: row['count'] for row in feature_counts_df.collect()}
class_totals        = class_counts_dict

log_prob_table = compute_log_probs(class_counts_dict, feature_counts_dict, class_totals)

print(f"Classes           : {log_prob_table['classes']}")
print(f"Log class probs   : {log_prob_table['log_class_probs']}")
print(f"Sample feat probs : {list(log_prob_table['log_feature_probs'].items())[:3]}")

In [ ]:
# Cell 5 — Prediction
#
# The predict UDF is still needed here — argmax over classes requires Python
# procedural logic that cannot be expressed as a Spark SQL expression.
# The optimisation over baseline is that we broadcast log_prob_table so it
# is sent once per executor instead of once per task.
bc_log_prob_table = sc.broadcast(log_prob_table)

# The UDF now accesses the broadcast value, not the driver-scope dict.
# Spark ships the broadcast variable once per executor; all tasks on that
# executor share the single cached copy in memory.
predict_udf = F.udf(
    lambda *features: predict(bc_log_prob_table.value, list(features)),
    StringType()
)

t_pred_start = time.time()

# Schema before: [buying, maint, doors, persons, lug_boot, safety, label]
# Schema after : [..., prediction]
prediction_df = test_df.withColumn(
    'prediction',
    predict_udf(*[F.col(c) for c in FEATURE_COLS])
)

results = prediction_df.select(LABEL_COL, 'prediction').collect()

pred_time = time.time() - t_pred_start
print(f'[TIMING] Prediction: {pred_time:.4f}s')
print(f'Sample: {results[:5]}')

In [ ]:
# Cell 6 — Evaluation

true_labels = [row[LABEL_COL] for row in results]
pred_labels = [row['prediction'] for row in results]

accuracy = evaluate(pred_labels, true_labels)

print(f'Accuracy     : {accuracy:.4f} ({accuracy * 100:.2f}%)')
print(f'Train time   : {train_time:.4f}s')
print(f'Predict time : {pred_time:.4f}s')
print(f'Total time   : {train_time + pred_time:.4f}s')

bc_log_prob_table.unpersist()
feature_counts_df.unpersist()

# TODO: Expected accuracy on the full car evaluation dataset is ~87.39%
# (Zheng 2014). With the 5-row dummy dataset accuracy is not meaningful.
# Replace DATA_PATH with the real file path before interpreting results.